In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
!pip install imbalanced-learn==0.11.0
!pip install torch-geometric==2.6.1
!pip install --upgrade scikit-learn
!pip install xgboost

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, BatchNorm, SAGEConv
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler

from sklearn.model_selection import train_test_split

from torch_geometric.datasets import Planetoid

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)


from sklearn.model_selection import KFold, train_test_split
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import torch_geometric

import os, glob, math
import networkx as nx
import seaborn as sns

In [ ]:

import zipfile
import os
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import glob
import math
import torch
import seaborn as sns
from torch_geometric.data import Data
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


knn = 3


zip_file_path = '/content/drive/MyDrive/1600_data/archive.zip'
extract_to_folder = 'X_edgelists'
GRAPH_DIR = '/content/X_edgelists/graphes_edgelist'
GRAPH_GLOB = '*.edgelist'
COMMUNITIES_CSV = '/content/drive/MyDrive/1600_data/film_categories.csv'
V1V2_CSV = '/content/drive/MyDrive/1600_data/V1V2.csv'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(extract_to_folder):
    os.makedirs(extract_to_folder)
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to_folder)


def load_film_graphs(graph_dir, pattern='*.edgelist'):
    paths = sorted(glob.glob(os.path.join(graph_dir, pattern)))
    film_graphs = {}
    for p in paths:
        film_id = os.path.splitext(os.path.basename(p))[0]
        try:
            g = nx.read_edgelist(p, nodetype=str, data=(('weight', float),), create_using=nx.Graph())
            film_graphs[film_id] = nx.Graph(g)
        except: continue
    return film_graphs

film_graphs = load_film_graphs(GRAPH_DIR, GRAPH_GLOB)
df_comm = pd.read_csv(COMMUNITIES_CSV)
film_to_community = dict(zip(df_comm['film_id'].astype(str), df_comm['category'].astype(str)))


films = [f for f in film_graphs.keys() if f in film_to_community]
films_clean = [str(f).strip() for f in films]



# D_METRIC = pd.read_csv('/content/drive/MyDrive/1600_data/1600_pairwise_Portrait_Divergence.csv')
D_METRIC = pd.read_csv('/content/drive/MyDrive/1600_data/1600_pairwise_NetLSD.csv')
# D_METRIC = pd.read_csv('/content/drive/MyDrive/1600_data/1600_pairwise_Laplacian.csv')


# DISTANCE_METRIC = "Portrait Divergence"
DISTANCE_METRIC = "NetLSD"
# DISTANCE_METRIC = "Laplacian"


W_METRIC = pd.read_csv('/content/drive/MyDrive/1600_data/1600_pairwise_Portrait_Divergence.csv')
# W_METRIC = pd.read_csv('/content/drive/MyDrive/1600_data/1600_pairwise_Laplacian.csv')
# W_METRIC = pd.read_csv('/content/drive/MyDrive/1600_data/1600_pairwise_NetLSD.csv')


WEIGHT_METRIC = "Portrait Divergence"
# WEIGHT_METRIC = "Laplacian"
# WEIGHT_METRIC = "NetLSD"



def prepare_matrix(df, films_list):
    df['SOURCE'] = df['SOURCE'].astype(str).str.strip()
    df['TARGET'] = df['TARGET'].astype(str).str.strip()
    pivot = df.pivot(index='SOURCE', columns='TARGET', values='distance')
    full = pivot.combine_first(pivot.T)
    mat = full.reindex(index=films_list, columns=films_list).values
    np.fill_diagonal(mat, 0)
    mat[np.isnan(mat)] = 1e6
    return mat


D_struct = prepare_matrix(D_METRIC, films_clean)
D_weights = prepare_matrix(W_METRIC, films_clean)


print(f"Moyenne {DISTANCE_METRIC}: {np.mean(D_struct[D_struct < 1e6]):.4f}")
print(f"Moyenne {WEIGHT_METRIC}: {np.mean(D_weights[D_weights < 1e6]):.4f}")


def build_hybrid_graph(films_list, D_s, D_w, mapping, k=knn):
    idx_map = {f: i for i, f in enumerate(films_list)}
    groups = {}
    for f in films_list: groups.setdefault(mapping[f], []).append(idx_map[f])

    G = nx.Graph()
    for f in films_list: G.add_node(f, community=mapping[f])

    for comm, ids in groups.items():
        ids = np.array(ids, dtype=int)
        if len(ids) <= 1: continue

        Dloc_s = D_s[np.ix_(ids, ids)]
        Dloc_w = D_w[np.ix_(ids, ids)]

        for a in range(len(ids)):

            order = np.argsort(Dloc_s[a])
            added = 0
            for b in order:
                if b == a or Dloc_s[a,b] >= 1e6: continue


                u, v = films_list[ids[a]], films_list[ids[b]]
                if not G.has_edge(u, v):

                    val_netlsd = float(Dloc_w[a,b])
                    if val_netlsd >= 1e6: val_netlsd = 1.0

                    G.add_edge(u, v, weight=val_netlsd, similarity=1.0/(1.0+val_netlsd))
                    edge_count = G.number_of_edges()

                added += 1
                if added >= k: break
    return G

G_films = build_hybrid_graph(films_clean, D_struct, D_weights, film_to_community, k=knn)


df_v1v2 = pd.read_csv(V1V2_CSV, delimiter=',')
films_in_feature_df = df_v1v2.iloc[:, 0].astype(str).tolist()

valid_films = [f for f in films_in_feature_df if f in G_films.nodes]
df_v1v2 = df_v1v2[df_v1v2.iloc[:, 0].astype(str).isin(valid_films)]
films_in_feature_df = df_v1v2.iloc[:, 0].astype(str).tolist()

X_raw = df_v1v2.iloc[:, 1:13].abs().fillna(0).values
comm_names = sorted(list(set(film_to_community.values())))
comm_map = {name: i for i, name in enumerate(comm_names)}
y_labels = np.array([comm_map[film_to_community[f]] for f in films_in_feature_df])

xgb_selector = xgb.XGBClassifier(
    max_depth=8,
    learning_rate=0.1,
    n_estimators=100,
    importance_type='gain',
    random_state=42
)

xgb_selector.fit(X_raw, y_labels)

selector = SelectFromModel(xgb_selector, max_features=6, threshold=-np.inf, prefit=True)
X_selected = selector.transform(X_raw)

X_scaled = MinMaxScaler().fit_transform(X_selected)


node_features = torch.tensor(X_scaled, dtype=torch.float)
node_map = {str(film): i for i, film in enumerate(films_in_feature_df)}

edge_index_list = []
edge_weight_list = []

for u, v, data in G_films.edges(data=True):
    u_s, v_s = str(u), str(v)
    if u_s in node_map and v_s in node_map:
        idx_u, idx_v = node_map[u_s], node_map[v_s]


        edge_index_list.extend([[idx_u, idx_v], [idx_v, idx_u]])


        sim_val = data.get('similarity', 1.0)
        edge_weight_list.extend([sim_val, sim_val])



def plot_graph_visualization(G):

    genre_labels = {
        0: 'Comedy', 1: 'Romance', 2: 'Drama', 3: 'Crime', 4: 'Adventure',
        5: 'Action', 7: 'Sci-fi', 8: 'Thriller', 10: 'Horror',
    }

    plt.figure(figsize=(15, 10))
    pos = nx.spring_layout(G, k=0.15, seed=42)


    nodes_comm_raw = [G.nodes[n]['community'] for n in G.nodes()]
    unique_comms = sorted(list(set(nodes_comm_raw)))


    cmap = plt.cm.get_cmap('viridis', len(unique_comms))
    node_colors = [cmap(unique_comms.index(G.nodes[n]['community'])) for n in G.nodes()]


    nx.draw_networkx_nodes(G, pos, node_size=60, node_color=node_colors, alpha=0.8)
    nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.2, edge_color='gray')


    from matplotlib.lines import Line2D
    legend_elements = []
    for i, comm_id in enumerate(unique_comms):

        label_name = genre_labels.get(int(comm_id), f"ID {comm_id}")

        legend_elements.append(
            Line2D([0], [0], marker='o', color='w', label=label_name,
                   markerfacecolor=cmap(i), markersize=12)
        )


    plt.legend(handles=legend_elements, title="Movie Genres",
               loc='center left', bbox_to_anchor=(1, 0.5),
               fontsize=12, title_fontsize=14, frameon=True)

    plt.title(f"{knn}-NN based on {DISTANCE_METRIC}", size=25, pad=20)
    plt.axis('off')


    plt.savefig(f'{knn}_NN based on {DISTANCE_METRIC}.png', dpi=600, bbox_inches='tight')
    plt.show()

plot_graph_visualization(G_films)


edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight_list, dtype=torch.float)

data = Data(
    x=node_features,
    edge_index=edge_index,
    edge_weight=edge_weight,
    edge_attr=edge_weight.view(-1, 1),
    y=torch.tensor(y_labels, dtype=torch.long)
)


print("\n--- RÉSUMÉ DU GRAPHE PyG ---")
print(data)
print(f"Exemple de poids d'arête (NetLSD): {data.edge_weight[0].item():.4f}")


plt.figure(figsize=(10, 8))
sorted_nodes = sorted(G_films.nodes(data=True), key=lambda x: x[1]['community'])
names = [n[0] for n in sorted_nodes]
adj = nx.to_pandas_adjacency(G_films, nodelist=names, weight='similarity')
sns.heatmap(adj, cmap="rocket_r", xticklabels=False, yticklabels=False)
plt.title(f"Matrix Intra-Community (KNN: {DISTANCE_METRIC} | Weights: {W_METRIC})")
plt.show()

In [ ]:
nombre_de_composants = nx.number_connected_components(G_films)
print(f"Le graphe est divisé en {nombre_de_composants} bloc(s).")

In [ ]:
def make_mask(idx, n):
    mask = torch.zeros(n, dtype=torch.bool)
    mask[idx] = True
    return mask

def train_one_epoch(model, data, optimizer, loss_fn):
    model.train()
    optimizer.zero_grad()
    logits = model(data)
    loss = loss_fn(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss.item())

@torch.no_grad()
def evaluate(model, data, mask, loss_fn):
    model.eval()
    logits = model(data)
    loss = float(loss_fn(logits[mask], data.y[mask]).item())
    pred = logits[mask].argmax(dim=1)
    acc = float((pred == data.y[mask]).float().mean().item())
    return acc, loss, pred, logits[mask]

In [ ]:
k_folds = 5
epochs = 7500
lr = 1e-3
weight_decay = 5e-4

unique_classes = sorted(list(Counter(data.y.tolist()).keys()))
label_mapping = {old_label: new_label for new_label, old_label in enumerate(unique_classes)}
mapped_labels = torch.tensor([label_mapping[l.item()] for l in data.y], dtype=torch.long)
data.y = mapped_labels


class_counts = Counter(data.y.tolist())
total_samples = len(data.y)
weights = [total_samples / (len(unique_classes) * class_counts[i]) for i in range(len(unique_classes))]
weights = torch.tensor(weights, dtype=torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = weights.to(device)
data = data.to(device)



kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

In [ ]:
unique_classes = sorted(list(Counter(data.y.tolist()).keys()))

print(unique_classes)

In [ ]:
def calculate_specificity(y_true, y_pred):
    from sklearn.metrics import confusion_matrix
    import numpy as np
    cm = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(len(cm)):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fp + fn)

        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specs.append(spec)
    return np.mean(specs)

In [ ]:
save_dir = f'/content/drive/MyDrive/1600_data/Results/{DISTANCE_METRIC}/'

In [ ]:
from torch_geometric.nn import GAT as GAT_off
from sklearn.model_selection import KFold, train_test_split
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

class MyGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=1, heads=1):
        super(MyGAT, self).__init__()
        self.conv1 = GAT_off(in_channels, hidden_channels, num_layers, heads=heads)
        self.conv2 = GAT_off(hidden_channels * heads, hidden_channels, num_layers=num_layers, heads=heads)
        self.conv3 = GAT_off(hidden_channels * heads, hidden_channels, num_layers=num_layers, heads=heads)
        self.conv4 = GAT_off(hidden_channels * heads, hidden_channels, num_layers=num_layers, heads=heads)
        self.conv5 = GAT_off(hidden_channels * heads, out_channels, num_layers=num_layers, heads=heads, concat=False)

    def forward(self, data, return_attention_weights=False):
        x, edge_index, edge_weight = data.x, data.edge_index, data.edge_weight
        x = self.conv1(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv2(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv3(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv4(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv5(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        return F.log_softmax(x, dim=1)



fold_results = {
    'train_loss': [[] for _ in range(epochs)],
    'val_loss': [[] for _ in range(epochs)],
    'train_acc': [[] for _ in range(epochs)],
    'val_acc': [[] for _ in range(epochs)]
}

test_metrics = []


all_indices = range(len(data.y))
train_val_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=42,
    stratify=data.y.cpu().numpy()
)


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline_mem = torch.cuda.memory_allocated() / (1024 ** 2)
else:
    baseline_mem = 0
    print("⚠️ Attention: Aucun GPU détecté.")

start_time_total = time.time()


kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

for fold, (fold_train_sub_idx, fold_val_sub_idx) in enumerate(kf.split(train_val_idx)):
    print(f'\n--- Fold {fold + 1}/{k_folds} ---')


    real_train_idx = [train_val_idx[i] for i in fold_train_sub_idx]
    real_val_idx = [train_val_idx[i] for i in fold_val_sub_idx]


    train_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)
    val_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)
    test_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)


    train_mask[real_train_idx] = True
    val_mask[real_val_idx] = True
    test_mask[test_idx] = True


    data.train_mask = train_mask
    data.val_mask = val_mask
    data.test_mask = test_mask


    model = MyGAT(in_channels=data.num_node_features, hidden_channels=64, out_channels=len(unique_classes), num_layers=1, heads=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if 'weights' in globals() and weights is not None:
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(device))
    else:
        loss_fn = torch.nn.CrossEntropyLoss()

    best_val = -1.0
    best_state = None


    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, data, optimizer, loss_fn)

        tr_acc, _, _, _ = evaluate(model, data, data.train_mask, loss_fn)
        val_acc, val_loss, _, _ = evaluate(model, data, data.val_mask, loss_fn)

        fold_results['train_loss'][epoch-1].append(train_loss)
        fold_results['val_loss'][epoch-1].append(val_loss)
        fold_results['train_acc'][epoch-1].append(tr_acc)
        fold_results['val_acc'][epoch-1].append(val_acc)


        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 500 == 0:
            print(f'  Epoch {epoch:4d} | Loss: {train_loss:.4f} | Train Acc: {tr_acc:.3f} | Val Acc: {val_acc:.3f}')


    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})


    test_acc, test_loss, test_pred, test_logits = evaluate(model, data, data.test_mask, loss_fn)
    print(f'>> Test Accuracy pour le Fold {fold + 1} : {test_acc:.4f}')

    test_true = data.y[data.test_mask].detach().cpu().numpy()
    test_pred_np = test_pred.detach().cpu().numpy()
    test_proba = torch.softmax(test_logits, dim=1).detach().cpu().numpy()

    acc = accuracy_score(test_true, test_pred_np)
    prec = precision_score(test_true, test_pred_np, average='macro', zero_division=0)
    sensitivity = recall_score(test_true, test_pred_np, average='macro', zero_division=0)
    f1 = f1_score(test_true, test_pred_np, average='macro', zero_division=0)
    auc = roc_auc_score(test_true, test_proba, multi_class='ovr')
    cm = confusion_matrix(test_true, test_pred_np)
    specificity = calculate_specificity(test_true, test_pred_np)

    test_metrics.append({
        'fold': fold + 1,
        'test_acc': acc,
        'test_precision_macro': prec,
        'sensitivity': sensitivity,
        'test_f1_macro': f1,
        'test_auc_ovr': auc,
        'specificity': specificity,
        'confusion_matrix': cm
    })


end_time_total = time.time()
total_duration_min = (end_time_total - start_time_total) / 60


df = pd.DataFrame([{k: v for k, v in m.items() if k != 'confusion_matrix'} for m in test_metrics])
display(df)


avg_train_loss = [np.mean(fold_results['train_loss'][e]) for e in range(epochs)]
avg_val_loss = [np.mean(fold_results['val_loss'][e]) for e in range(epochs)]
avg_train_acc = [np.mean(fold_results['train_acc'][e]) for e in range(epochs)]
avg_val_acc = [np.mean(fold_results['val_acc'][e]) for e in range(epochs)]

plt.figure(figsize=(14, 5))
plt.suptitle('GAT', fontsize=25, fontweight='bold')

plt.subplot(1, 2, 1)
plt.plot(avg_train_loss, label='Train Loss', color='#1f77b4')
plt.plot(avg_val_loss, label='Val Loss', color='#ff7f0e')
plt.title('Average Training and Validation Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(avg_train_acc, label='Train Acc', color='#1f77b4')
plt.plot(avg_val_acc, label='Val Acc', color='#ff7f0e')
plt.title('Average Training and Validation Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plot_filename = f"{DISTANCE_METRIC}_{WEIGHT_METRIC}_GAT.png"
plot_path = os.path.join(save_dir, plot_filename)
plt.savefig(plot_path, dpi=600, bbox_inches='tight')
plt.show()


print('\n================================')
print(f'Average Test Accuracy          : {df["test_acc"].mean():.4f}')
print(f'Average Test Precision (macro) : {df["test_precision_macro"].mean():.4f}')
print(f'Average Test Sensitivity       : {df["sensitivity"].mean():.4f}')
print(f'Average Test F1 (macro)        : {df["test_f1_macro"].mean():.4f}')
print(f'Average Test AUC (ovr)         : {df["test_auc_ovr"].mean():.4f}')
print(f'Average Test Specificity       : {df["specificity"].mean():.4f}')
print(f'Temps d\'exécution total       : {total_duration_min:.2f} min')


summary_values = df.select_dtypes(include=[np.number]).mean().to_frame().T
summary_values.index = [str(DISTANCE_METRIC)]
summary_values['Total_Time'] = total_duration_min
csv_filename = os.path.join(save_dir, f"{DISTANCE_METRIC}_{WEIGHT_METRIC}_GAT_Performance_Metrics.csv")
summary_values.to_csv(csv_filename, index=True)
print(f"Metrics file saved : {csv_filename}")

In [ ]:
from torch_geometric.nn import GAT as GAT_off
from sklearn.model_selection import KFold, train_test_split
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from torch_geometric.nn import GCNConv

class MyGCNConv(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, improved=True):
        super(MyGCNConv, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels, improved=improved)
        self.conv2 = GCNConv(hidden_channels, hidden_channels, improved=improved)
        self.conv3 = GCNConv(hidden_channels, hidden_channels, improved=improved)
        self.conv4 = GCNConv(hidden_channels, hidden_channels, improved=improved)
        self.conv5 = GCNConv(hidden_channels, out_channels, improved=improved)

    def forward(self, data, **kwargs):

        x, edge_index, edge_weight = data.x, data.edge_index, data.edge_weight

        x = self.conv1(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv2(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv3(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv4(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv5(x, edge_index, edge_weight)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        return F.log_softmax(x, dim=1)


fold_results = {
    'train_loss': [[] for _ in range(epochs)],
    'val_loss': [[] for _ in range(epochs)],
    'train_acc': [[] for _ in range(epochs)],
    'val_acc': [[] for _ in range(epochs)]
}

test_metrics = []


all_indices = range(len(data.y))
train_val_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=42,
    stratify=data.y.cpu().numpy()
)


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline_mem = torch.cuda.memory_allocated() / (1024 ** 2)
else:
    baseline_mem = 0
    print("⚠️ Attention: Aucun GPU détecté.")

start_time_total = time.time()


kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

for fold, (fold_train_sub_idx, fold_val_sub_idx) in enumerate(kf.split(train_val_idx)):
    print(f'\n--- Fold {fold + 1}/{k_folds} ---')


    real_train_idx = [train_val_idx[i] for i in fold_train_sub_idx]
    real_val_idx = [train_val_idx[i] for i in fold_val_sub_idx]


    train_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)
    val_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)
    test_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)


    train_mask[real_train_idx] = True
    val_mask[real_val_idx] = True
    test_mask[test_idx] = True


    data.train_mask = train_mask
    data.val_mask = val_mask
    data.test_mask = test_mask


    model = MyGCNConv(in_channels=data.num_node_features, hidden_channels=64, out_channels=len(unique_classes), improved=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if 'weights' in globals() and weights is not None:
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(device))
    else:
        loss_fn = torch.nn.CrossEntropyLoss()

    best_val = -1.0
    best_state = None


    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, data, optimizer, loss_fn)

        tr_acc, _, _, _ = evaluate(model, data, data.train_mask, loss_fn)
        val_acc, val_loss, _, _ = evaluate(model, data, data.val_mask, loss_fn)

        fold_results['train_loss'][epoch-1].append(train_loss)
        fold_results['val_loss'][epoch-1].append(val_loss)
        fold_results['train_acc'][epoch-1].append(tr_acc)
        fold_results['val_acc'][epoch-1].append(val_acc)


        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 500 == 0:
            print(f'  Epoch {epoch:4d} | Loss: {train_loss:.4f} | Train Acc: {tr_acc:.3f} | Val Acc: {val_acc:.3f}')


    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})


    test_acc, test_loss, test_pred, test_logits = evaluate(model, data, data.test_mask, loss_fn)
    print(f'>> Test Accuracy pour le Fold {fold + 1} : {test_acc:.4f}')

    test_true = data.y[data.test_mask].detach().cpu().numpy()
    test_pred_np = test_pred.detach().cpu().numpy()
    test_proba = torch.softmax(test_logits, dim=1).detach().cpu().numpy()

    acc = accuracy_score(test_true, test_pred_np)
    prec = precision_score(test_true, test_pred_np, average='macro', zero_division=0)
    sensitivity = recall_score(test_true, test_pred_np, average='macro', zero_division=0)
    f1 = f1_score(test_true, test_pred_np, average='macro', zero_division=0)
    auc = roc_auc_score(test_true, test_proba, multi_class='ovr')
    cm = confusion_matrix(test_true, test_pred_np)
    specificity = calculate_specificity(test_true, test_pred_np)

    test_metrics.append({
        'fold': fold + 1,
        'test_acc': acc,
        'test_precision_macro': prec,
        'sensitivity': sensitivity,
        'test_f1_macro': f1,
        'test_auc_ovr': auc,
        'specificity': specificity,
        'confusion_matrix': cm
    })


end_time_total = time.time()
total_duration_min = (end_time_total - start_time_total) / 60


df = pd.DataFrame([{k: v for k, v in m.items() if k != 'confusion_matrix'} for m in test_metrics])
display(df)


avg_train_loss = [np.mean(fold_results['train_loss'][e]) for e in range(epochs)]
avg_val_loss = [np.mean(fold_results['val_loss'][e]) for e in range(epochs)]
avg_train_acc = [np.mean(fold_results['train_acc'][e]) for e in range(epochs)]
avg_val_acc = [np.mean(fold_results['val_acc'][e]) for e in range(epochs)]

plt.figure(figsize=(14, 5))
plt.suptitle('GCN', fontsize=25, fontweight='bold')

plt.subplot(1, 2, 1)
plt.plot(avg_train_loss, label='Train Loss', color='#1f77b4')
plt.plot(avg_val_loss, label='Val Loss', color='#ff7f0e')
plt.title('Average Training and Validation Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(avg_train_acc, label='Train Acc', color='#1f77b4')
plt.plot(avg_val_acc, label='Val Acc', color='#ff7f0e')
plt.title('Average Training and Validation Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plot_filename = f"{DISTANCE_METRIC}_{WEIGHT_METRIC}_GCN.png"
plot_path = os.path.join(save_dir, plot_filename)
plt.savefig(plot_path, dpi=600, bbox_inches='tight')
plt.show()


print('\n================================')
print(f'Average Test Accuracy          : {df["test_acc"].mean():.4f}')
print(f'Average Test Precision (macro) : {df["test_precision_macro"].mean():.4f}')
print(f'Average Test Sensitivity       : {df["sensitivity"].mean():.4f}')
print(f'Average Test F1 (macro)        : {df["test_f1_macro"].mean():.4f}')
print(f'Average Test AUC (ovr)         : {df["test_auc_ovr"].mean():.4f}')
print(f'Average Test Specificity       : {df["specificity"].mean():.4f}')
print(f'Temps d\'exécution total       : {total_duration_min:.2f} min')


summary_values = df.select_dtypes(include=[np.number]).mean().to_frame().T
summary_values.index = [str(DISTANCE_METRIC)]
summary_values['Total_Time'] = total_duration_min
csv_filename = os.path.join(save_dir, f"{DISTANCE_METRIC}_{WEIGHT_METRIC}_GCN_Performance_Metrics.csv")
summary_values.to_csv(csv_filename, index=True)
print(f"Metrics file saved: {csv_filename}")

In [ ]:
from torch_geometric.nn import GAT as GAT_off
from sklearn.model_selection import KFold, train_test_split
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

class MyGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.2):
        super(MyGraphSAGE, self).__init__()
        self.dropout = dropout


        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.conv4 = SAGEConv(hidden_channels, hidden_channels)
        self.conv5 = SAGEConv(hidden_channels, out_channels)

    def forward(self, data, **kwargs):

        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv4(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv5(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        return F.log_softmax(x, dim=1)



fold_results = {
    'train_loss': [[] for _ in range(epochs)],
    'val_loss': [[] for _ in range(epochs)],
    'train_acc': [[] for _ in range(epochs)],
    'val_acc': [[] for _ in range(epochs)]
}

test_metrics = []


all_indices = range(len(data.y))
train_val_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=42,
    stratify=data.y.cpu().numpy()
)


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline_mem = torch.cuda.memory_allocated() / (1024 ** 2)
else:
    baseline_mem = 0
    print("⚠️ Attention: Aucun GPU détecté.")

start_time_total = time.time()


kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

for fold, (fold_train_sub_idx, fold_val_sub_idx) in enumerate(kf.split(train_val_idx)):
    print(f'\n--- Fold {fold + 1}/{k_folds} ---')


    real_train_idx = [train_val_idx[i] for i in fold_train_sub_idx]
    real_val_idx = [train_val_idx[i] for i in fold_val_sub_idx]


    train_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)
    val_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)
    test_mask = torch.zeros(len(data.y), dtype=torch.bool, device=device)


    train_mask[real_train_idx] = True
    val_mask[real_val_idx] = True
    test_mask[test_idx] = True


    data.train_mask = train_mask
    data.val_mask = val_mask
    data.test_mask = test_mask


    model = MyGraphSAGE(in_channels=data.num_node_features, hidden_channels=64, out_channels=len(unique_classes), dropout=0.5).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if 'weights' in globals() and weights is not None:
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(device))
    else:
        loss_fn = torch.nn.CrossEntropyLoss()

    best_val = -1.0
    best_state = None


    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, data, optimizer, loss_fn)

        tr_acc, _, _, _ = evaluate(model, data, data.train_mask, loss_fn)
        val_acc, val_loss, _, _ = evaluate(model, data, data.val_mask, loss_fn)

        fold_results['train_loss'][epoch-1].append(train_loss)
        fold_results['val_loss'][epoch-1].append(val_loss)
        fold_results['train_acc'][epoch-1].append(tr_acc)
        fold_results['val_acc'][epoch-1].append(val_acc)


        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 500 == 0:
            print(f'  Epoch {epoch:4d} | Loss: {train_loss:.4f} | Train Acc: {tr_acc:.3f} | Val Acc: {val_acc:.3f}')


    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})


    test_acc, test_loss, test_pred, test_logits = evaluate(model, data, data.test_mask, loss_fn)
    print(f'>> Test Accuracy pour le Fold {fold + 1} : {test_acc:.4f}')

    test_true = data.y[data.test_mask].detach().cpu().numpy()
    test_pred_np = test_pred.detach().cpu().numpy()
    test_proba = torch.softmax(test_logits, dim=1).detach().cpu().numpy()

    acc = accuracy_score(test_true, test_pred_np)
    prec = precision_score(test_true, test_pred_np, average='macro', zero_division=0)
    sensitivity = recall_score(test_true, test_pred_np, average='macro', zero_division=0)
    f1 = f1_score(test_true, test_pred_np, average='macro', zero_division=0)
    auc = roc_auc_score(test_true, test_proba, multi_class='ovr')
    cm = confusion_matrix(test_true, test_pred_np)
    specificity = calculate_specificity(test_true, test_pred_np)

    test_metrics.append({
        'fold': fold + 1,
        'test_acc': acc,
        'test_precision_macro': prec,
        'sensitivity': sensitivity,
        'test_f1_macro': f1,
        'test_auc_ovr': auc,
        'specificity': specificity,
        'confusion_matrix': cm
    })


end_time_total = time.time()
total_duration_min = (end_time_total - start_time_total) / 60


df = pd.DataFrame([{k: v for k, v in m.items() if k != 'confusion_matrix'} for m in test_metrics])
display(df)


avg_train_loss = [np.mean(fold_results['train_loss'][e]) for e in range(epochs)]
avg_val_loss = [np.mean(fold_results['val_loss'][e]) for e in range(epochs)]
avg_train_acc = [np.mean(fold_results['train_acc'][e]) for e in range(epochs)]
avg_val_acc = [np.mean(fold_results['val_acc'][e]) for e in range(epochs)]

plt.figure(figsize=(14, 5))
plt.suptitle('GraphSAGE', fontsize=16, fontweight='bold')

plt.subplot(1, 2, 1)
plt.plot(avg_train_loss, label='Train Loss', color='#1f77b4')
plt.plot(avg_val_loss, label='Val Loss', color='#ff7f0e')
plt.title('Average Training and Validation Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(avg_train_acc, label='Train Acc', color='#1f77b4')
plt.plot(avg_val_acc, label='Val Acc', color='#ff7f0e')
plt.title('Average Training and Validation Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

plot_filename = f"{DISTANCE_METRIC}_{WEIGHT_METRIC}_GraphSAGE.png"
plot_path = os.path.join(save_dir, plot_filename)
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()


print('\n=================================================')
print(f'Average Test Accuracy          : {df["test_acc"].mean():.4f}')
print(f'Average Test Precision (macro) : {df["test_precision_macro"].mean():.4f}')
print(f'Average Test Sensitivity       : {df["sensitivity"].mean():.4f}')
print(f'Average Test F1 (macro)        : {df["test_f1_macro"].mean():.4f}')
print(f'Average Test AUC (ovr)         : {df["test_auc_ovr"].mean():.4f}')
print(f'Average Test Specificity       : {df["specificity"].mean():.4f}')
print(f'Temps d\'exécution total       : {total_duration_min:.2f} min')


summary_values = df.select_dtypes(include=[np.number]).mean().to_frame().T
summary_values.index = [str(DISTANCE_METRIC)]
summary_values['Total_Time'] = total_duration_min
csv_filename = os.path.join(save_dir, f"{DISTANCE_METRIC}_{WEIGHT_METRIC}_GraphSAGE_Performance_Metrics.csv")
summary_values.to_csv(csv_filename, index=True)
print(f"Metrics file saved : {csv_filename}")